# Laboratório: Agentes baseados em objetivos e Métodos de Busca I

## Ajustando ambiente

In [ ]:
%%capture

import matplotlib.pyplot as plt
import networkx as nx

## Motivação

O estudo de agentes baseados em objetivos e métodos de busca é
fundamental na Inteligência Artificial, pois capacita os sistemas a
planejarem com antecedência e a simularem sequências de ações que formam
um caminho até um estado objetivo quando a atitude correta a ser tomada
não é imediatamente óbvia. Nesse cenário, compreender a **busca não
informada** é de extrema importância e forma o alicerce da resolução de
problemas, já que essas estratégias operam sem qualquer conhecimento ou
pista do domínio sobre a proximidade de um estado em relação ao
objetivo. Ao estudar métodos não informados como a busca em largura, de
custo uniforme e em profundidade, compreendemos como é possível explorar
sistematicamente espaços de estados imensos ou infinitos de forma a
garantir propriedades críticas, como a completude (encontrar uma solução
se ela existir) e a otimização de custos. Além disso, a busca não
informada nos força a lidar diretamente com o maior desafio da
exploração de caminhos: a explosão combinatória e a complexidade de
tempo e espaço. Estratégias não informadas, como a busca em profundidade
e o aprofundamento iterativo, são consideradas as ferramentas de
trabalho básicas em muitas áreas da IA justamente devido ao seu uso
parcimonioso e inteligente da memória, ensinando-nos princípios
essenciais de estruturas de dados e limites computacionais que preparam
o terreno para atuar em ambientes desconhecidos e para o desenvolvimento
de buscas heurísticas mais avançadas.

## Objetivos de Aprendizagem

- **Entender agentes baseados em objetivos:** compreender como esses
  agentes formulam metas e buscam planos para alcançá-las.
- **Modelar problemas de busca:** definir estado inicial, objetivo,
  ações, transições e custos.
- **Entender a busca em largura (BFS):** como ela explora o espaço de
  estados nível por nível, garantindo completude e o menor número de
  passos.
- **Entender a busca de custo uniforme (UCS):** como ela generaliza a
  BFS para grafos com custos de aresta diferentes, expandindo sempre o
  caminho de menor custo acumulado.
- **Entender a busca em profundidade (DFS):** suas vantagens em uso de
  memória e suas limitações.
- **Comparar BFS, UCS e DFS:** em termos de estratégia de exploração,
  completude e otimalidade.
- **Implementar os algoritmos:** diretamente sobre um grafo representado
  como dicionário, sem camadas de abstração adicionais entre o estado e
  a lógica de busca.

## Implementação

### Representação do ambiente

O ambiente é um grafo simples: um dicionário onde cada cidade aponta
para suas vizinhas (com o custo de cada aresta). Não precisamos de
nenhuma classe para representá-lo — `environment[cidade]` já devolve os
vizinhos.

In [ ]:
environment = {
    'Natal': {'Parnamirim': 1, 'Extremoz': 1, 'São Gonçalo do Amarante': 1, 'Macaíba': 1},
    'Parnamirim': {'Natal': 1, 'São José de Mipibu': 1, 'Macaíba': 1},
    'Extremoz': {'Natal': 1},
    'São Gonçalo do Amarante': {'Natal': 1, 'Macaíba': 1, 'Ceará-Mirim': 1},
    'Macaíba': {'Natal': 1, 'Parnamirim': 1, 'São Gonçalo do Amarante': 1, 'Ielmo Marinho': 1, 'Vera Cruz': 1},
    'Ceará-Mirim': {'São Gonçalo do Amarante': 1},
    'Ielmo Marinho': {'Macaíba': 1},
    'Vera Cruz': {'Macaíba': 1, 'Monte Alegre': 1},
    'Monte Alegre': {'Vera Cruz': 1, 'São José de Mipibu': 1},
    'São José de Mipibu': {'Parnamirim': 1, 'Goianinha': 1, 'Monte Alegre': 1},
    'Goianinha': {'São José de Mipibu': 1, 'Tibau do Sul': 1},
    'Tibau do Sul': {'Goianinha': 1, 'Pipa': 1},
    'Pipa': {'Tibau do Sul': 1}
}

Para visualizar esse grafo, usamos o [NetworkX](https://networkx.org/):
montamos um `nx.DiGraph()` a partir do dicionário `environment` (uma
aresta para cada vizinho de cada cidade) e desenhamos com `nx.draw`.
Essa visualização é só para referência — os algoritmos de busca abaixo
trabalham diretamente com o dicionário, sem depender do NetworkX.

In [ ]:
G = nx.DiGraph()
for origem, destinos in environment.items():
    for destino in destinos:
        G.add_edge(origem, destino)

pos = nx.spring_layout(G, seed=7)

plt.figure(figsize=(10, 6))
nx.draw(
    G,
    pos,
    with_labels=True,
    node_color='#d9e8fb',
    node_size=1800,
    arrows=True,
    font_size=11,
    edgecolors='black'
)
plt.title('Estrutura do grafo do ambiente')
plt.show()

### Funções auxiliares

Para a busca em si, cada candidato na fronteira é **o caminho percorrido
até ali** (uma lista de cidades, ex: `['Natal', 'Macaíba']`), em vez de
um objeto `Node` com ponteiro para o pai. O estado atual é sempre o
último elemento do caminho (`path[-1]`), e o caminho já É a solução —
não precisa ser reconstruído no final.

In [ ]:
def path_cost(graph, path):
    """Soma o custo das arestas percorridas em um caminho."""
    return sum(graph[a][b] for a, b in zip(path, path[1:]))

`path_cost` soma o custo de cada aresta percorrida no caminho. Neste
capítulo todas as arestas custam 1 (ver `environment` acima), então o
custo de um caminho é simplesmente o seu comprimento.

## Exemplo prático

Agora vamos aplicar essas funções em um cenário concreto, com um estado
inicial `Natal` e um objetivo `Pipa`.

In [ ]:
graph = environment
estado_inicial = 'Natal'
objetivo = 'Pipa'

print('Estado inicial:', estado_inicial)
print('Objetivo:', objetivo)
print('Ações possíveis em Natal:', list(graph[estado_inicial].keys()))

### Busca em largura

A busca em largura é uma estratégia de busca na qual o nó raiz é
expandido primeiro, seguido por todos os seus nós sucessores, depois os
sucessores destes, e assim por diante, explorando o espaço de estados
nível por nível. O algoritmo geralmente utiliza uma estrutura de dados
do tipo fila FIFO (primeiro a entrar, primeiro a sair), o que garante
que os nós recém-gerados (mais profundos) vão para o final da fila e os
nós mais antigos e rasos sejam expandidos primeiro. Por explorar todas
as opções de uma profundidade antes de avançar para a próxima, a busca
em largura é sistemática, completa e garante encontrar uma solução com o
número mínimo de ações necessárias.

In [ ]:
def breadth_first_search(graph, start, goal, return_expansion_order=False):
    """Busca em largura: expande o caminho mais raso primeiro (fila FIFO)."""
    if start == goal:
        return ([start], [start]) if return_expansion_order else [start]

    frontier = [[start]]
    visited = {start}
    expansion_order = []

    while frontier:
        path = frontier.pop(0)
        current = path[-1]
        expansion_order.append(current)

        for neighbor in graph.get(current, {}):
            if neighbor in visited:
                continue
            new_path = path + [neighbor]
            if neighbor == goal:
                expansion_order.append(neighbor)
                return (new_path, expansion_order) if return_expansion_order else new_path
            visited.add(neighbor)
            frontier.append(new_path)

    return (None, expansion_order) if return_expansion_order else None

In [ ]:
resultado_bfs, expansoes_bfs = breadth_first_search(graph, estado_inicial, objetivo, return_expansion_order=True)
print('Caminho encontrado (BFS):', resultado_bfs)
print('Ordem de expansão BFS:', expansoes_bfs)

### Busca em profundidade

A busca em profundidade é uma estratégia que expande sempre o nó mais
profundo disponível na fronteira de busca. O algoritmo avança
imediatamente para o nível mais profundo da árvore e, ao alcançar um nó
sem sucessores, ele “retrocede” para o próximo nó mais profundo que
ainda possui opções não exploradas. Geralmente implementada com uma
estrutura de dados do tipo pilha (último a entrar, primeiro a sair),
essa abordagem destaca-se pelo uso muito reduzido de memória. Contudo,
ao contrário da busca em largura, ela não garante encontrar a solução de
menor custo e pode ser incompleta em espaços de estados infinitos, pois
corre o risco de ficar presa indefinidamente em um único caminho.

In [ ]:
def depth_first_search(graph, start, goal, return_expansion_order=False):
    """Busca em profundidade: expande o caminho mais recente primeiro (pilha)."""
    frontier = [[start]]
    visited = set()
    expansion_order = []

    while frontier:
        path = frontier.pop()
        current = path[-1]
        expansion_order.append(current)

        if current == goal:
            return (path, expansion_order) if return_expansion_order else path

        if current in visited:
            continue
        visited.add(current)

        for neighbor in graph.get(current, {}):
            if neighbor not in visited:
                frontier.append(path + [neighbor])

    return (None, expansion_order) if return_expansion_order else None

In [ ]:
resultado_dfs, expansoes_dfs = depth_first_search(graph, estado_inicial, objetivo, return_expansion_order=True)
print('Caminho encontrado (DFS):', resultado_dfs)
print('Ordem de expansão DFS:', expansoes_dfs)

## Busca uniforme

A busca de custo uniforme (*Uniform Cost Search* ou UCS) é uma
generalização da BFS para grafos com custos de aresta diferentes: em vez
de expandir sempre o caminho mais raso, ela expande sempre o caminho de
**menor custo acumulado** (`g(n)`). Quando todas as arestas custam o
mesmo (como no `environment` deste capítulo), UCS e BFS produzem o mesmo
resultado — a diferença só aparece em grafos com pesos variados, como o
`environment_with_distance` do próximo capítulo.

In [ ]:
def uniform_cost_search(graph, start, goal, return_expansion_order=False):
    """Busca de custo uniforme: expande sempre o caminho de menor custo acumulado."""
    frontier = [[start]]
    visited = set()
    expansion_order = []

    while frontier:
        frontier.sort(key=lambda path: (path_cost(graph, path), path[-1]))
        path = frontier.pop(0)
        current = path[-1]

        if current in visited:
            continue
        visited.add(current)
        expansion_order.append(current)

        if current == goal:
            return (path, expansion_order) if return_expansion_order else path

        for neighbor in graph.get(current, {}):
            if neighbor not in visited:
                frontier.append(path + [neighbor])

    return (None, expansion_order) if return_expansion_order else None

In [ ]:
resultado_ucs, expansoes_ucs = uniform_cost_search(graph, estado_inicial, objetivo, return_expansion_order=True)
print('Caminho encontrado (UCS):', resultado_ucs)
print('Ordem de expansão UCS:', expansoes_ucs)

## Desafio

### Busca em profundidade limitada

Implemente uma versão da **busca em profundidade limitada**. Nessa
estratégia, a exploração segue em profundidade, mas o algoritmo só pode
expandir nós até uma profundidade máxima definida por `limit`.

Sua função deve:

1.  Começar no estado inicial;
2.  Explorar os caminhos em profundidade (mesma lógica de pilha da DFS);
3.  Impedir a expansão de caminhos com profundidade maior ou igual ao
    limite;
4.  Retornar o caminho solução se encontrar o objetivo;
5.  Retornar `None` se nenhuma solução for encontrada dentro do limite.

A profundidade de um caminho é `len(path) - 1` (o número de arestas
percorridas).

In [ ]:
def depth_limited_search(graph, start, goal, limit=2):
    """Busca em profundidade sem expandir caminhos além da profundidade `limit`."""
    # Comece com o caminho inicial na fronteira.
    frontier = [[start]]

    while frontier:
        path = frontier.pop()
        current = path[-1]

        # Verifique se o objetivo foi encontrado.
        if current == goal:
            return path

        # Expanda o caminho apenas se ele ainda estiver dentro do limite.
        if len(path) - 1 < limit:
            for neighbor in graph.get(current, {}):
                # Adicione os novos caminhos à fronteira.
                pass

    return None

### Busca em profundidade iterativa

A **busca em profundidade iterativa** (*Iterative Deepening Search*,
IDS) combina as vantagens da BFS e da DFS: repete a busca em
profundidade limitada com limites crescentes (0, 1, 2, …) até encontrar
o objetivo, obtendo o caminho mais curto (como a BFS) sem precisar
manter toda a fronteira em memória de uma vez (como a DFS).

Sua função deve:

1.  Tentar limites de `0` até `max_limit`, um de cada vez;
2.  Para cada limite, chamar
    `depth_limited_search(graph, start, goal, limit)`;
3.  Retornar o primeiro caminho encontrado;
4.  Retornar `None` se nenhum limite até `max_limit` encontrar o
    objetivo.

In [ ]:
def iterative_deepening_search(graph, start, goal, max_limit=10):
    """Busca em profundidade iterativa: chama depth_limited_search com limites crescentes."""
    for limit in range(max_limit + 1):
        # Tente encontrar o objetivo com o limite atual.
        pass

    return None

Dicas

Uma implementação possível de `depth_limited_search`:

``` python
def depth_limited_search(graph, start, goal, limit=2):
    frontier = [[start]]

    while frontier:
        path = frontier.pop()
        current = path[-1]

        if current == goal:
            return path

        if len(path) - 1 < limit:
            for neighbor in graph.get(current, {}):
                frontier.append(path + [neighbor])

    return None
```

E `iterative_deepening_search`, reaproveitando a função acima:

``` python
def iterative_deepening_search(graph, start, goal, max_limit=10):
    for limit in range(max_limit + 1):
        result = depth_limited_search(graph, start, goal, limit)
        if result is not None:
            return result

    return None
```

## Perguntas para reflexão

1.  Por que a BFS testa o objetivo ao *gerar* um vizinho, enquanto a DFS
    e a UCS testam ao *retirar* o caminho da fronteira? O que
    aconteceria se a BFS testasse na retirada, ou se a DFS testasse na
    geração?
2.  Em que tipo de grafo a UCS encontraria um caminho diferente do
    encontrado pela BFS? Por que, no `environment` deste capítulo, as
    duas sempre coincidem?
3.  Por que representar cada candidato da fronteira como o caminho
    completo (lista de cidades) elimina a necessidade de uma estrutura
    como `Node` com ponteiro para o nó pai?
4.  Qual a relação entre a busca em profundidade limitada e a busca em
    profundidade iterativa? Por que a IDS é completa mesmo
    reaproveitando uma busca que, sozinha, não é?

## Key takeaways

- Um algoritmo de busca explora um grafo representando cada candidato
  como o caminho percorrido até ele — o estado atual é sempre o último
  elemento do caminho.
- BFS e DFS usam a mesma representação de grafo, mas exploram o espaço
  de estados de formas diferentes (fila vs. pilha) e testam o objetivo
  em momentos diferentes (na geração vs. na retirada).
- A solução já é o próprio caminho encontrado — não é preciso
  reconstruir nada a partir de ponteiros de pai.

## Referencias

1.  Russell, S. & Norvig, P. (2010). Artificial Intelligence: A Modern
    Approach. Prentice Hall.